# Prerequisites
본 `ipynb` 은 `Python=3.12` 에서 작성하였습니다. Package dependency 를 해결하기 위해 아래 cell 을 실행해주세요.

## Install Python packages

In [ ]:
%pip -q install -U agent-framework==1.0.0b260130

## Load environment variables

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

# How to handle Memory of Agent Context

메모리 파일의 기본 구조를 정의합니다.

In [ ]:
from pathlib import Path
from datetime import datetime

MEMORY_FILE = Path("./resources/MEMORY.md")

# 초기 메모리 파일 생성
initial_memory = """# Agent Memory

이 파일은 에이전트의 장기 메모리를 저장합니다.

## User Preferences
- 주 사용 언어: 
- 코딩 스타일: PEP8 준수

## Important Facts
- 사용자 이름: 
- 프로젝트: Azure AI 학습

## Conversation History Summary
(대화 요약이 여기에 추가됩니다)

## Notes
(중요한 메모가 여기에 추가됩니다)
"""

if not MEMORY_FILE.exists():
    MEMORY_FILE.write_text(initial_memory, encoding="utf-8")
    print(f"✅ MEMORY.md 파일이 생성되었습니다.")
else:
    print(f"📄 MEMORY.md 파일이 이미 존재합니다.")

print(f"\n현재 메모리 내용:\n{'-'*40}")
print(MEMORY_FILE.read_text(encoding="utf-8"))

Agent 의 응답을 출력하는 함수를 정의합니다.

In [ ]:
from agent_framework import Role

def print_result(result):
    """에이전트 실행 결과를 포맷팅하여 출력"""
    
    # 도구 호출 정보 출력
    print("\n======= 🤖 Agent Messages =======")
    for idx,msg in enumerate(result.messages):
        if msg.role == Role.ASSISTANT:
            for content in msg.contents:
                if content.type == "text":
                    print(f"[{idx}] 💬 {content.text}")
                elif content.type == "function_call":
                    print(f"[{idx}] 🔧 {content.type} | {content.name} | {content.raw_representation.arguments}")
                else:
                    print(f"[{idx}] not supported content type: {content.type}")
        elif msg.role == Role.TOOL:
            for content in msg.contents:
                print(f"[{idx}] 📤 {content.result}")
        else:
            print(f"[{idx}] not supported role: {msg.role}")

## Use Tools
에이전트가 부여받은 Tools 을 이용하여 Token Generating 에서 Function Calling 으로 메모리를 증강시키는 방법입니다. 에이전트가 사용할 메모리 관련 도구들을 정의합니다.

In [ ]:
from typing import Annotated
from pydantic import Field
from agent_framework import tool

@tool(name="read_memory", description="MEMORY.md 파일에서 저장된 메모리/컨텍스트를 읽어옵니다. 대화 시작 시 또는 이전 정보가 필요할 때 사용하세요.")
def read_memory() -> str:
    """메모리 파일 전체를 읽어옵니다."""
    try:
        if not MEMORY_FILE.exists():
            return "❌ 메모리 파일이 존재하지 않습니다. 새로운 대화를 시작합니다."

        content = MEMORY_FILE.read_text(encoding="utf-8")
        return f"✅ 저장된 메모리:\n{content}"
    except Exception as e:
        return f"❌ 메모리 읽기 오류: {str(e)}"

@tool(name="read_memory_section", description="MEMORY.md 파일에서 특정 섹션만 읽어옵니다.")
def read_memory_section(
    section: Annotated[str, Field(description="읽을 섹션 이름 (예: 'User Preferences', 'Important Facts', 'Notes')")]
) -> str:
    """메모리 파일에서 특정 섹션만 추출합니다."""
    try:
        if not MEMORY_FILE.exists():
            return "❌ 메모리 파일이 존재하지 않습니다."

        content = MEMORY_FILE.read_text(encoding="utf-8")
        lines = content.split("\n")
        section_content = []
        in_section = False
        for line in lines:
            if line.startswith("## ") and section.lower() in line.lower():
                in_section = True
                section_content.append(line)
            elif line.startswith("## ") and in_section:
                break
            elif in_section:
                section_content.append(line)

        if section_content:
            return "\n".join(section_content)
        return f"❌ '{section}' 섹션을 찾을 수 없습니다."
    except Exception as e:
        return f"❌ 섹션 읽기 오류: {str(e)}"

@tool(name="save_memory", description="MEMORY.md 파일의 특정 섹션에 새로운 정보를 추가/수정합니다.")
def save_memory(
    section: Annotated[str, Field(description="저장할 섹션 이름 (예: 'User Preferences', 'Important Facts', 'Notes')")],
    content: Annotated[str, Field(description="추가/수정할 내용 (예: '사용자 이름: 이진호')")]
) -> str:
    """메모리 파일의 특정 섹션에 내용을 추가/수정합니다. 동일 key가 있으면 값만 업데이트하고, 없으면 새로 추가합니다."""
    def _extract_key(content: str) -> str:
        """'key: value' 형식에서 key 추출 (예: '사용자 이름: 이진호' -> '사용자 이름')"""
        if ":" in content:
            return content.split(":", 1)[0].strip()
        return content.strip()

    def _line_matches_key(line: str, key: str) -> bool:
        """'- key: ...' 형식의 라인이 해당 key와 일치하는지 확인"""
        if not line.strip().startswith("-"):
            return False
        rest = line.strip()[1:].strip()
        return rest.startswith(key + ":") if rest else False

    try:
        if not MEMORY_FILE.exists():
            return "❌ 메모리 파일이 존재하지 않습니다."
        
        memory_content = MEMORY_FILE.read_text(encoding="utf-8")
        lines = memory_content.split("\n")
        new_lines = []
        section_found = False
        in_section = False
        content_updated = False
        key = _extract_key(content)
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
        new_line = f"- {content} [{timestamp}]"
        
        for idx, line in enumerate(lines):
            # 섹션 시작인지 체크
            if line.startswith("## ") and section.lower() in line.lower():
                section_found = True
                in_section = True
                new_lines.append(line)
                continue

            # 이미 섹션 안에 있으면서 새 섹션 시작
            if in_section and line.startswith("## "):
                if not content_updated:
                    new_lines.append(new_line)
                    content_updated = True
                # 다음 섹션 전에 빈 줄 추가
                if new_lines and new_lines[-1].strip() != "":
                    new_lines.append("")
                new_lines.append(line)
                in_section = False
                continue

            # 섹션 안: 동일 key가 있으면 값만 업데이트, 없으면 기존 라인 유지
            if in_section:
                if _line_matches_key(line, key):
                    new_lines.append(new_line)
                    content_updated = True
                else:
                    new_lines.append(line)
                continue

            new_lines.append(line)
        
        # 섹션이 파일 끝까지 이어지는 경우
        if in_section and not content_updated:
            new_lines.append(new_line)
            content_updated = True
        
        if not section_found:
            return f"❌ '{section}' 섹션을 찾을 수 없습니다."

        MEMORY_FILE.write_text("\n".join(new_lines), encoding="utf-8")
        return f"✅ '{section}' 섹션에 메모리가 저장되었습니다."
    except Exception as e:
        return f"❌ 메모리 저장 오류: {str(e)}"

@tool(name="update_memory", description="MEMORY.md 파일의 특정 내용을 수정합니다.")
def update_memory(
    old_content: Annotated[str, Field(description="수정할 기존 내용")],
    new_content: Annotated[str, Field(description="새로운 내용")]
) -> str:
    """메모리 파일의 특정 내용을 수정합니다."""
    try:
        if not MEMORY_FILE.exists():
            return "❌ 메모리 파일이 존재하지 않습니다."
        
        content = MEMORY_FILE.read_text(encoding="utf-8")
        if old_content not in content:
            return f"❌ '{old_content}'를 찾을 수 없습니다."

        updated_content = content.replace(old_content, new_content)
        MEMORY_FILE.write_text(updated_content, encoding="utf-8")
        return f"✅ 메모리가 수정되었습니다: '{old_content}' → '{new_content}'"
    except Exception as e:
        return f"❌ 메모리 수정 오류: {str(e)}"

# 도구 목록
memory_tools = [read_memory, read_memory_section, save_memory, update_memory]
print("✅ Memory Tools 정의 완료:", [t.name for t in memory_tools])

메모리 도구를 사용하는 에이전트를 생성합니다.

In [ ]:
from agent_framework import ChatAgent
from agent_framework.openai import OpenAIResponsesClient

# 메모리 에이전트 생성
memory_agent = ChatAgent(
    chat_client=OpenAIResponsesClient(
        base_url=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        model_id=AZURE_OPENAI_CHAT_DEPLOYMENT,
    ),
    name="MemoryAgent",
    instructions="""당신은 장기 메모리를 가진 AI 어시스턴트입니다.

중요한 규칙:
1. 대화 시작 시 반드시 read_memory 도구로 이전 메모리를 확인하세요.
2. 사용자에 대해 새로운 정보를 알게 되면 save_memory로 저장하세요.
3. 사용자 선호도, 중요한 사실, 대화 요약 등을 적극적으로 기억하세요.
4. 이전 대화에서 기억한 정보를 활용하여 개인화된 응답을 제공하세요.

메모리 섹션:
- User Preferences: 사용자 선호도 (언어, 스타일 등)
- Important Facts: 사용자에 대한 중요 정보
- Notes: 기타 메모""",
    tools=memory_tools,
)

### 테스트: 메모리 시스템 동작 확인

In [ ]:
# 테스트 1: 메모리 읽기 테스트
thread = memory_agent.get_new_thread()

result = await memory_agent.run(
    "안녕하세요! 저에 대해 기억하고 있는 게 있나요?",
    thread=thread
)
print_result(result)

In [ ]:
# 테스트 2: 새로운 정보 저장
result = await memory_agent.run(
    "제 이름은 이진호이고, 저는 Python을 주로 사용합니다. 이 정보를 기억해주세요.",
    thread=thread
)
print_result(result)

In [ ]:
# 테스트 3: 저장된 메모리 확인
print("현재 MEMORY.md 내용:")
print("=" * 40)
print(MEMORY_FILE.read_text(encoding="utf-8"))

In [ ]:
# 테스트 4: 새 대화 시작 - 메모리 활용
new_thread = memory_agent.get_new_thread()

result = await memory_agent.run(
    "제 이름이 뭐였죠? 어떤 프로그래밍 언어를 선호하나요?",
    thread=new_thread
)
print_result(result)

In [ ]:
# 테스트 5: 새 대화 시작 - 메모리 활용
result = await memory_agent.run(
    "생각해보니 Java 도 좋아해요. 이 정보도 기억해주세요.",
    thread=new_thread
)
print_result(result)

## Use Prompt
다른 방법으로, 대화 시작 시 메모리를 Prompt에 자동으로 포함시킬 수 있습니다.

In [ ]:
from agent_framework import ChatAgent
from agent_framework.openai import OpenAIResponsesClient

# 메모리 파일 읽기
memory_content = ""
if MEMORY_FILE.exists():
    memory_content = MEMORY_FILE.read_text(encoding="utf-8")

# 메모리가 포함된 시스템 프롬프트
system_prompt = f"""당신은 장기 메모리를 가진 AI 어시스턴트입니다.

=== 저장된 메모리 ===
{memory_content}
=== 메모리 끝 ===

위 메모리를 참고하여 사용자에게 개인화된 응답을 제공하세요.
새로운 중요 정보가 있으면 save_memory 도구로 저장하세요."""
    
memory_agent = ChatAgent(
    chat_client=OpenAIResponsesClient(
        base_url=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        model_id=AZURE_OPENAI_CHAT_DEPLOYMENT,
    ),
    name="SmartMemoryAgent",
    instructions=system_prompt,
)

### 테스트: 메모리 시스템 동작 확인

In [ ]:
# 테스트: 메모리가 자동 주입된 에이전트
thread = memory_agent.get_new_thread()

result = await memory_agent.run(
    "제 이름과 사용하는 언어가 뭐였죠?",
    thread=thread
)
print_result(result)

## Wrap Up

### 메모리 시스템 구현 방법

| 방법 | 설명 | 장점 | 단점 |
|------|------|------|------|
| **도구 기반** | `read_memory` 도구로 동적 로드 | 필요할 때만 로드, 유연함 | 매번 도구 호출 필요 |
| **Prompt 주입** | 대화 시작 시 메모리 전체 주입 | 즉시 사용 가능, 빠름 | 토큰 사용량 증가 |
| **하이브리드** | 핵심 정보는 주입 + 상세 정보는 도구 | 균형 잡힌 접근 | 구현 복잡 |

# Cleanup

In [ ]:
# MEMORY.md 파일 삭제 (선택사항)
if MEMORY_FILE.exists():
    MEMORY_FILE.unlink()
    print("MEMORY.md 파일이 삭제되었습니다.")